# LeetCode 502 — IPO bằng Python

- Link: https://leetcode.com/problems/ipo/
- Độ khó: **Hard**
- Chủ đề: Greedy (tham lam), Heap (max-heap), Sắp xếp, Hai con trỏ

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findMaximizedCapital(self, k: int, w: int, profits: List[int], capital: List[int]) -> int:

```

## 1. Mục tiêu học

1. Đọc hiểu một bài **tối ưu theo lượt**: mỗi lượt chọn một dự án, vốn tăng dần, tập dự án "mở khóa" lớn dần.
2. Nhận ra và **chứng minh** chiến lược tham lam: *mỗi lượt chọn dự án lợi nhuận lớn nhất trong số làm được*.
3. Dùng **sắp xếp theo vốn yêu cầu + con trỏ** để "mở khóa" dự án, và **max-heap** để lấy lợi nhuận lớn nhất nhanh.
4. So sánh 4 cách: vét cạn, tham lam quét tuyến tính, sắp xếp + heap, và tối ưu "đường tắt" khi mọi dự án đã làm được.
5. Kiểm chứng tham lam bằng vét cạn trên dữ liệu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Công ty sắp IPO và muốn tăng vốn bằng cách làm **tối đa `k` dự án khác nhau** trước khi IPO. Có `n` dự án; dự án `i` cần **vốn tối thiểu** `capital[i]` để bắt đầu và khi xong đem lại **lợi nhuận thuần** `profits[i]` (cộng vào vốn). Vốn ban đầu là `w`.

Làm dự án **không tiêu tốn** vốn: chỉ cần vốn hiện tại `>= capital[i]` là được làm, xong thì vốn tăng thêm `profits[i]`. Mỗi dự án làm tối đa một lần.

Trả về **vốn lớn nhất** có thể đạt được.

**Ví dụ 1**

```text
Input:  k = 2, w = 0, profits = [1,2,3], capital = [0,1,1]
Output: 4
Giải thích: vốn 0 chỉ làm được dự án 0 -> vốn 1.
            Vốn 1 làm được dự án 1 hoặc 2; chọn dự án 2 (lợi nhuận 3) -> vốn 4.
```

**Ví dụ 2**

```text
Input:  k = 3, w = 0, profits = [1,2,3], capital = [0,1,2]
Output: 6
Giải thích: 0 -> làm dự án 0 (vốn 1) -> dự án 1 (vốn 3) -> dự án 2 (vốn 6).
```

**Ràng buộc**

- `1 <= k <= 10^5`
- `0 <= w <= 10^9`
- `n == profits.length == capital.length`, `1 <= n <= 10^5`
- `0 <= profits[i] <= 10^4`
- `0 <= capital[i] <= 10^9`

In [1]:
k, w = 2, 0
profits, capital = [1, 2, 3], [0, 1, 1]
for i in range(len(profits)):
    print(f"dự án {i}: cần vốn {capital[i]}, lợi nhuận {profits[i]}, làm được với w={w}? {capital[i] <= w}")

dự án 0: cần vốn 0, lợi nhuận 1, làm được với w=0? True
dự án 1: cần vốn 1, lợi nhuận 2, làm được với w=0? False
dự án 2: cần vốn 1, lợi nhuận 3, làm được với w=0? False


## 3. Kiến thức nền

### 3.1. `zip` và sắp xếp danh sách cặp

`zip(capital, profits)` ghép hai danh sách thành các cặp `(capital[i], profits[i])`. `sorted(...)` sắp các cặp theo phần tử đầu (vốn), bằng nhau thì theo phần tử sau.

In [2]:
capital = [5, 0, 3, 0]
profits = [10, 1, 7, 4]
projects = sorted(zip(capital, profits))
print(projects)

[(0, 1), (0, 4), (3, 7), (5, 10)]


### 3.2. Max-heap bằng `heapq` và số đối

`heapq` là min-heap. Đẩy `-profit` vào thì đỉnh là lợi nhuận **lớn nhất** (âm nhỏ nhất). Lấy ra nhớ đổi dấu lại.

In [3]:
import heapq
h = []
for p in [3, 10, 1, 7]:
    heapq.heappush(h, -p)
print("lớn nhất:", -heapq.heappop(h), "| tiếp theo:", -heapq.heappop(h))

lớn nhất: 10 | tiếp theo: 7


### 3.3. Con trỏ "mở khóa" trên danh sách đã sắp xếp

Khi vốn `w` tăng dần, tập dự án làm được (`capital <= w`) là một **tiền tố** của danh sách đã sắp theo vốn và tiền tố đó chỉ dài ra. Một con trỏ `i` chỉ tiến, không lùi → tổng công sức `O(n)` cho mọi lần mở khóa.

In [4]:
projects = sorted(zip([5, 0, 3, 0], [10, 1, 7, 4]))
i = 0
for w in [0, 2, 4, 9]:
    while i < len(projects) and projects[i][0] <= w:
        i += 1
    print(f"w={w}: mở khóa tới chỉ số {i} -> {projects[:i]}")

w=0: mở khóa tới chỉ số 2 -> [(0, 1), (0, 4)]
w=2: mở khóa tới chỉ số 2 -> [(0, 1), (0, 4)]
w=4: mở khóa tới chỉ số 3 -> [(0, 1), (0, 4), (3, 7)]
w=9: mở khóa tới chỉ số 4 -> [(0, 1), (0, 4), (3, 7), (5, 10)]


### 3.4. `heapq.nlargest`

`heapq.nlargest(k, iterable)` trả về `k` phần tử lớn nhất trong `O(n log k)`. Dùng cho tối ưu "đường tắt" ở Cách 4.

In [5]:
print(heapq.nlargest(2, [3, 10, 1, 7]), sum(heapq.nlargest(2, [3, 10, 1, 7])))

[10, 7] 17


### 3.5. `itertools.permutations` (cho vét cạn)

`permutations(items, r)` sinh mọi cách chọn **có thứ tự** `r` phần tử phân biệt. Số lượng tăng như giai thừa — chỉ dùng cho input rất nhỏ.

In [6]:
from itertools import permutations
print(list(permutations([0, 1, 2], 2)))

[(0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1)]


## 4. Ý tưởng nền tảng / trực giác

Tính tay với `k = 2`, `w = 1`, `profits = [5, 2, 9]`, `capital = [1, 0, 6]`:

- Vốn 1: làm được dự án 0 (cần 1, lãi 5) và dự án 1 (cần 0, lãi 2). Dự án 2 cần 6 — chưa được.
- Nếu chọn dự án 1 trước (lãi 2): vốn 3 → vẫn chưa đủ 6, làm dự án 0 → vốn 8. Tổng: 8.
- Nếu chọn dự án 0 trước (lãi 5): vốn 6 → **mở khóa** dự án 2 (lãi 9) → vốn 15. Tổng: 15.

Trực giác: **lãi càng lớn càng tốt**, vì vốn lớn hơn không bao giờ làm mất lựa chọn nào — chỉ mở thêm lựa chọn. Lợi nhuận không âm nên làm thêm dự án không bao giờ hại.

In [7]:
def simulate(order, w, profits, capital):
    for i in order:
        if capital[i] > w:
            return None          # thứ tự không hợp lệ
        w += profits[i]
    return w

profits, capital = [5, 2, 9], [1, 0, 6]
for order in permutations(range(3), 2):
    print(order, "->", simulate(order, 1, profits, capital))

(0, 1) -> 8
(0, 2) -> 15
(1, 0) -> 8
(1, 2) -> None
(2, 0) -> None
(2, 1) -> None


## 5. Cách 1 — Brute Force: thử mọi thứ tự chọn

**Ý tưởng.** DFS: ở mỗi lượt, thử mọi dự án chưa làm mà vốn hiện tại đủ; đệ quy với vốn mới; giữ vốn lớn nhất gặp được (kể cả dừng sớm). Đây là đáp án "chắc chắn đúng" dùng để kiểm chứng tham lam.

In [8]:
def ipo_brute(k, w, profits, capital):
    n = len(profits)
    best = w
    used = [False] * n

    def dfs(left, cur):
        nonlocal best
        best = max(best, cur)
        if left == 0:
            return
        for i in range(n):
            if not used[i] and capital[i] <= cur:
                used[i] = True
                dfs(left - 1, cur + profits[i])
                used[i] = False

    dfs(k, w)
    return best

print(ipo_brute(2, 0, [1, 2, 3], [0, 1, 1]), ipo_brute(3, 0, [1, 2, 3], [0, 1, 2]))
assert ipo_brute(2, 0, [1, 2, 3], [0, 1, 1]) == 4
assert ipo_brute(3, 0, [1, 2, 3], [0, 1, 2]) == 6
assert ipo_brute(2, 1, [5, 2, 9], [1, 0, 6]) == 15

4 6


**Độ phức tạp:** `O(n^k)` (chỉnh hợp) — với `n = 10`, `k = 5` đã là `30 240` nhánh; với ràng buộc thật là bất khả thi. Space `O(k)` stack.

**Nút thắt:** ta thử cả những dự án lãi nhỏ dù đang có dự án lãi lớn hơn làm được. Nếu chứng minh được "lãi lớn nhất luôn là lựa chọn tốt" thì mỗi lượt chỉ còn một nhánh.

In [9]:
import time
t = time.perf_counter()
ipo_brute(5, 0, list(range(1, 11)), [0] * 10)
print(f"n=10, k=5: {time.perf_counter() - t:.3f}s")

n=10, k=5: 0.004s


## 6. Bước đột phá

**Insight (tham lam):** Ở mỗi lượt, trong các dự án **làm được** (`capital <= w`) và chưa làm, chọn dự án có **lợi nhuận lớn nhất**.

**Vì sao không hại? (lập luận trao đổi — exchange argument)**
- Gọi `g` là dự án lãi lớn nhất làm được ở lượt hiện tại. Xét một chiến lược tối ưu `OPT` không chọn `g` ở lượt này mà chọn `x` (với `profits[x] <= profits[g]`).
- Nếu `OPT` có dùng `g` ở một lượt sau: đổi chỗ `x` và `g`. Vốn sau lượt hiện tại **không giảm** (`+profits[g] >= +profits[x]`), và mọi lượt ở giữa vẫn hợp lệ vì vốn chỉ lớn hơn hoặc bằng trước. Tới lượt cũ của `g`, ta làm `x` — `x` làm được từ đầu nên càng làm được. Vốn cuối như cũ.
- Nếu `OPT` không dùng `g`: thay `x` bằng `g`. Vốn từ đây trở đi lớn hơn hoặc bằng → mọi dự án sau vẫn hợp lệ → vốn cuối không giảm.
- Vậy luôn có một lời giải tối ưu bắt đầu bằng `g`. Áp dụng lặp lại cho các lượt sau.

**Hai tính chất "đơn điệu" then chốt:**
1. Lợi nhuận `>= 0` → vốn không giảm.
2. Vốn không giảm → tập dự án làm được chỉ **lớn dần** (một khi mở khóa thì luôn mở).

**Quy tắc:** mỗi lượt: **mở khóa** mọi dự án `capital <= w` vào "rổ ứng viên"; **nếu** rổ rỗng **thì** dừng; ngược lại lấy dự án lãi lớn nhất.

In [10]:
# Kiểm chứng nhanh insight trên ví dụ tự nghĩ: chọn lãi lớn nhất mỗi lượt
def greedy_trace(k, w, profits, capital):
    used = set()
    for t in range(k):
        avail = [i for i in range(len(profits)) if i not in used and capital[i] <= w]
        if not avail:
            break
        g = max(avail, key=lambda i: profits[i])
        used.add(g); w += profits[g]
        print(f"lượt {t+1}: làm được {avail}, chọn {g} (lãi {profits[g]}) -> vốn {w}")
    return w

greedy_trace(2, 1, [5, 2, 9], [1, 0, 6])

lượt 1: làm được [0, 1], chọn 0 (lãi 5) -> vốn 6
lượt 2: làm được [1, 2], chọn 2 (lãi 9) -> vốn 15


15

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tham lam, quét tuyến tính mỗi lượt

Mỗi lượt quét toàn bộ `n` dự án tìm dự án chưa làm, làm được, lãi lớn nhất.

In [11]:
def ipo_scan(k, w, profits, capital):
    n = len(profits)
    used = [False] * n
    for _ in range(k):
        best_i = -1
        for i in range(n):
            if not used[i] and capital[i] <= w and (best_i == -1 or profits[i] > profits[best_i]):
                best_i = i
        if best_i == -1:
            break
        used[best_i] = True
        w += profits[best_i]
    return w

print(ipo_scan(2, 0, [1, 2, 3], [0, 1, 1]), ipo_scan(3, 0, [1, 2, 3], [0, 1, 2]))

4 6


**Code nộp LeetCode (Cách 2)** — đúng nhưng `O(k·n)` sẽ TLE khi `k = n = 10^5`:

In [12]:
from typing import List

class SolutionScan:
    def findMaximizedCapital(self, k: int, w: int, profits: List[int], capital: List[int]) -> int:
        return ipo_scan(k, w, profits, capital)

print(SolutionScan().findMaximizedCapital(2, 0, [1, 2, 3], [0, 1, 1]))

4


**Độ phức tạp Cách 2:** Time `O(k·n)` = `10^10` xấu nhất. Space `O(n)`.

### 7.2. Cách 3 — Sắp xếp theo vốn + max-heap (lời giải chính)

1. `projects = sorted(zip(capital, profits))`.
2. Con trỏ `i = 0`, heap rỗng.
3. Lặp `k` lần:
   - Trong khi `projects[i].capital <= w`: đẩy `-profit` vào heap, `i += 1` (**mở khóa**).
   - Nếu heap rỗng: dừng (không còn dự án làm được).
   - `w += -heappop(heap)`.

In [13]:
def ipo_heap(k, w, profits, capital):
    projects = sorted(zip(capital, profits))
    heap = []
    i, n = 0, len(projects)
    for _ in range(k):
        while i < n and projects[i][0] <= w:
            heapq.heappush(heap, -projects[i][1])
            i += 1
        if not heap:
            break
        w -= heapq.heappop(heap)        # cộng lợi nhuận lớn nhất
    return w

print(ipo_heap(2, 0, [1, 2, 3], [0, 1, 1]), ipo_heap(3, 0, [1, 2, 3], [0, 1, 2]))
assert ipo_heap(2, 1, [5, 2, 9], [1, 0, 6]) == 15

4 6


**Code nộp LeetCode (Cách 3):** xem mục 9.

**Độ phức tạp Cách 3:** sắp xếp `O(n log n)`; mỗi dự án vào heap tối đa 1 lần `O(log n)`; tối đa `k` lần pop `O(log n)` → Time `O(n log n + k log n)`. Space `O(n)`.

### 7.3. Cách 4 — Heap + "đường tắt" khi mọi dự án đã làm được

Quan sát: nếu `w >= max(capital)` thì mọi dự án đều làm được ngay từ đầu, và đáp án đơn giản là `w + tổng k lợi nhuận lớn nhất`. Tương tự, trong khi chạy Cách 3, nếu con trỏ `i` đã mở khóa hết (`i == n`), các lượt còn lại chỉ việc lấy các phần tử lớn nhất còn trong heap — có thể cộng một lần bằng `nlargest`. Đây là tối ưu hằng số, hữu ích khi `k` rất lớn.

In [14]:
def ipo_shortcut(k, w, profits, capital):
    if w >= max(capital):
        return w + sum(heapq.nlargest(k, profits))
    projects = sorted(zip(capital, profits))
    heap = []
    i, n = 0, len(projects)
    while k > 0:
        while i < n and projects[i][0] <= w:
            heapq.heappush(heap, -projects[i][1])
            i += 1
        if i == n:                                     # đã mở khóa tất cả
            return w + sum(-x for x in heapq.nsmallest(k, heap))
        if not heap:
            break
        w -= heapq.heappop(heap)
        k -= 1
    return w

print(ipo_shortcut(2, 0, [1, 2, 3], [0, 1, 1]), ipo_shortcut(3, 0, [1, 2, 3], [0, 1, 2]), ipo_shortcut(2, 5, [1, 2, 3], [0, 1, 1]))

4 6 10


**Code nộp LeetCode (Cách 4):**

In [15]:
class SolutionShortcut:
    def findMaximizedCapital(self, k: int, w: int, profits: List[int], capital: List[int]) -> int:
        return ipo_shortcut(k, w, profits, capital)

print(SolutionShortcut().findMaximizedCapital(3, 0, [1, 2, 3], [0, 1, 2]))

6


**Độ phức tạp Cách 4:** xấu nhất như Cách 3 `O((n + k) log n)`; trường hợp `w >= max(capital)` chỉ `O(n log k)`.

## 8. TRACE từng bước

Trace **Cách 3** với `k = 3`, `w = 0`, `profits = [1, 2, 3, 5]`, `capital = [0, 1, 1, 4]`.

`projects` sau khi sắp: `[(0,1), (1,2), (1,3), (4,5)]`.

| lượt | w trước | mở khóa thêm | heap (lãi) | chọn | w sau |
|---|---|---|---|---|---|
| 1 | 0 | (0,1) | {1} | 1 | 1 |
| 2 | 1 | (1,2), (1,3) | {2, 3} | 3 | 4 |
| 3 | 4 | (4,5) | {2, 5} | 5 | 9 |

In [16]:
def trace_heap(k, w, profits, capital):
    projects = sorted(zip(capital, profits))
    print("projects:", projects)
    heap, i, n = [], 0, len(projects)
    for t in range(1, k + 1):
        unlocked = []
        while i < n and projects[i][0] <= w:
            heapq.heappush(heap, -projects[i][1]); unlocked.append(projects[i]); i += 1
        if not heap:
            print(f"lượt {t}: w={w}, không còn dự án làm được -> dừng")
            break
        before = w
        pick = -heapq.heappop(heap)
        w += pick
        print(f"lượt {t}: w={before:>2} | mở khóa {unlocked} | heap sau khi lấy={sorted(-x for x in heap)} | chọn lãi {pick} -> w={w}")
    return w

trace_heap(3, 0, [1, 2, 3, 5], [0, 1, 1, 4])

projects: [(0, 1), (1, 2), (1, 3), (4, 5)]
lượt 1: w= 0 | mở khóa [(0, 1)] | heap sau khi lấy=[] | chọn lãi 1 -> w=1
lượt 2: w= 1 | mở khóa [(1, 2), (1, 3)] | heap sau khi lấy=[2] | chọn lãi 3 -> w=4
lượt 3: w= 4 | mở khóa [(4, 5)] | heap sau khi lấy=[2] | chọn lãi 5 -> w=9


9

Trace trường hợp **bị kẹt** (heap rỗng giữa chừng):

In [17]:
trace_heap(4, 0, [1, 1, 10], [0, 0, 5])

projects: [(0, 1), (0, 1), (5, 10)]
lượt 1: w= 0 | mở khóa [(0, 1), (0, 1)] | heap sau khi lấy=[1] | chọn lãi 1 -> w=1
lượt 2: w= 1 | mở khóa [] | heap sau khi lấy=[] | chọn lãi 1 -> w=2
lượt 3: w=2, không còn dự án làm được -> dừng


2

## 9. Code nộp LeetCode

In [18]:
import heapq
from typing import List

class Solution:
    def findMaximizedCapital(self, k: int, w: int, profits: List[int], capital: List[int]) -> int:
        projects = sorted(zip(capital, profits))     # sắp theo vốn yêu cầu
        heap = []                                    # max-heap lợi nhuận (lưu số âm)
        i, n = 0, len(projects)
        for _ in range(k):
            while i < n and projects[i][0] <= w:     # mở khóa mọi dự án đủ vốn
                heapq.heappush(heap, -projects[i][1])
                i += 1
            if not heap:                             # không làm được dự án nào nữa
                break
            w -= heapq.heappop(heap)                 # làm dự án lãi lớn nhất
        return w

sol = Solution()
assert sol.findMaximizedCapital(2, 0, [1, 2, 3], [0, 1, 1]) == 4
assert sol.findMaximizedCapital(3, 0, [1, 2, 3], [0, 1, 2]) == 6
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đơn điệu).** Vì `profits[i] >= 0`, vốn sau mỗi lượt không giảm; do đó tập `A(w) = {i : capital[i] <= w}` chỉ lớn dần.

**Bổ đề 2 (lựa chọn tham lam an toàn).** Ở trạng thái (vốn `w`, tập đã dùng `U`, còn `r` lượt), gọi `g` là dự án lãi lớn nhất trong `A(w) \ U`. Tồn tại lời giải tối ưu cho trạng thái này chọn `g` đầu tiên. (Chứng minh bằng trao đổi như mục 6: thay/hoán đổi lựa chọn đầu của `OPT` bằng `g` không làm giảm vốn ở bất kỳ thời điểm nào sau đó, nên mọi lựa chọn sau vẫn hợp lệ.)

**Bổ đề 3 (heap chứa đúng `A(w) \ U`).** Con trỏ `i` đẩy vào heap mọi dự án có `capital <= w` (theo thứ tự tăng dần của vốn), mỗi dự án đúng một lần; dự án đã làm bị pop ra. Vậy heap = `A(w) \ U`, và đỉnh heap là `g`.

**Kết luận.** Quy nạp theo số lượt: thuật toán luôn đi theo một lời giải tối ưu. Khi heap rỗng, không còn dự án nào làm được ở vốn hiện tại và vốn không thể tăng nữa → dừng là đúng. Làm ít hơn `k` dự án không phải vấn đề vì đề cho phép "tối đa `k`".

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n^k)` | `O(k)` |
| 2. Quét mỗi lượt | `O(k·n)` | `O(n)` |
| 3. Sắp xếp + heap | `O(n log n + k log n)` | `O(n)` |
| 4. Heap + đường tắt | như 3; `O(n log k)` khi `w >= max(capital)` | `O(n)` |

In [19]:
import random
random.seed(502)
n = 100000
P = [random.randint(0, 10**4) for _ in range(n)]
C = [random.randint(0, 10**9) for _ in range(n)]
for name, f in [("heap", ipo_heap), ("shortcut", ipo_shortcut), ("Solution", Solution().findMaximizedCapital)]:
    t = time.perf_counter()
    res = f(n, 10**8, P, C)
    print(f"{name:<9}: {time.perf_counter() - t:.3f}s -> {res}")
t = time.perf_counter()
ipo_scan(300, 10**8, P, C)
print(f"scan (chỉ k=300): {time.perf_counter() - t:.3f}s  -> k=10^5 sẽ chậm hơn ~300 lần")

heap     : 0.051s -> 199083478


shortcut : 0.052s -> 199083478


Solution : 0.053s -> 199083478


scan (chỉ k=300): 1.101s  -> k=10^5 sẽ chậm hơn ~300 lần


## 12. Bộ test

In [20]:
methods = {
    "brute": ipo_brute,
    "scan": ipo_scan,
    "heap": ipo_heap,
    "shortcut": ipo_shortcut,
    "SolScan": SolutionScan().findMaximizedCapital,
    "SolShortcut": SolutionShortcut().findMaximizedCapital,
    "Solution": Solution().findMaximizedCapital,
}
cases = [
    ((2, 0, [1, 2, 3], [0, 1, 1]), 4),
    ((3, 0, [1, 2, 3], [0, 1, 2]), 6),
    ((1, 0, [1, 2, 3], [1, 1, 2]), 0),          # không làm được gì
    ((10, 0, [1, 2, 3], [0, 1, 2]), 6),         # k > n
    ((2, 1, [5, 2, 9], [1, 0, 6]), 15),         # chọn lãi lớn để mở khóa
    ((1, 2, [1, 2, 3], [1, 1, 2]), 5),          # mọi dự án làm được: lấy lãi lớn nhất
    ((3, 0, [0, 0, 0], [0, 0, 0]), 0),          # lãi bằng 0
    ((2, 0, [1, 1, 10], [0, 0, 5]), 2),         # bị kẹt: không đủ vốn cho dự án 10
    ((1, 10**9, [10**4], [10**9]), 10**9 + 10**4),
]
for (k, w, P_, C_), exp in cases:
    for name, f in methods.items():
        got = f(k, w, list(P_), list(C_))
        assert got == exp, (name, k, w, P_, C_, got, exp)
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [21]:
random.seed(2026)
for trial in range(800):
    n = random.randint(1, 7)
    k = random.randint(1, 5)
    w = random.randint(0, 6)
    P_ = [random.randint(0, 6) for _ in range(n)]
    C_ = [random.randint(0, 12) for _ in range(n)]
    exp = ipo_brute(k, w, P_, C_)
    for name, f in methods.items():
        got = f(k, w, list(P_), list(C_))
        assert got == exp, (name, k, w, P_, C_, got, exp)
print("800 test ngẫu nhiên (seed 2026): tham lam khớp vét cạn")

800 test ngẫu nhiên (seed 2026): tham lam khớp vét cạn


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chọn dự án vốn yêu cầu nhỏ nhất (hoặc theo tỉ lệ lãi/vốn)

**Code sai:** sắp theo `capital` rồi làm lần lượt `k` dự án đầu tiên.

**Vì sao sai:** dự án rẻ chưa chắc lãi nhiều; bỏ lỡ dự án lãi lớn cũng làm được.

In [22]:
def ipo_cheapest(k, w, profits, capital):
    for c, p in sorted(zip(capital, profits))[:k]:
        if c <= w:
            w += p
    return w

print("sai:", ipo_cheapest(1, 5, [1, 100], [0, 5]), " đúng:", Solution().findMaximizedCapital(1, 5, [1, 100], [0, 5]))

sai: 6  đúng: 105


**Code đúng:** chọn lãi **lớn nhất** trong các dự án làm được.

### Lỗi 2 — Chọn lãi lớn nhất **toàn cục** mà quên kiểm tra vốn

**Code sai:** sắp `profits` giảm dần và cộng `k` số đầu.

**Vì sao sai:** dự án lãi lớn có thể cần vốn chưa đủ.

In [23]:
def ipo_ignore_capital(k, w, profits, capital):
    return w + sum(sorted(profits, reverse=True)[:k])

print("sai:", ipo_ignore_capital(2, 0, [1, 2, 3], [0, 1, 1]), " đúng:", Solution().findMaximizedCapital(2, 0, [1, 2, 3], [0, 1, 1]))

sai: 5  đúng: 4


### Lỗi 3 — Quên `break` khi heap rỗng (hoặc `heappop` trên heap rỗng)

**Code sai:** `w -= heapq.heappop(heap)` không kiểm tra → `IndexError` khi không còn dự án làm được.

In [24]:
def ipo_no_check(k, w, profits, capital):
    projects = sorted(zip(capital, profits)); heap = []; i = 0
    for _ in range(k):
        while i < len(projects) and projects[i][0] <= w:
            heapq.heappush(heap, -projects[i][1]); i += 1
        w -= heapq.heappop(heap)              # SAI: heap có thể rỗng
    return w

try:
    ipo_no_check(2, 0, [1, 1, 10], [0, 5, 5])
except IndexError as e:
    print("IndexError:", e)

IndexError: index out of range


**Code đúng:** `if not heap: break`.

### Lỗi 4 — Mở khóa chỉ một lần trước vòng lặp

**Code sai:** đẩy mọi dự án `capital <= w` ban đầu vào heap một lần, rồi chỉ pop `k` lần.

**Vì sao sai:** vốn tăng sau mỗi dự án sẽ mở khóa thêm dự án mới — phải mở khóa **trong mỗi lượt**.

In [25]:
def ipo_unlock_once(k, w, profits, capital):
    heap = [-p for p, c in zip(profits, capital) if c <= w]
    heapq.heapify(heap)
    for _ in range(k):
        if not heap:
            break
        w -= heapq.heappop(heap)
    return w

print("sai:", ipo_unlock_once(3, 0, [1, 2, 3], [0, 1, 2]), " đúng:", Solution().findMaximizedCapital(3, 0, [1, 2, 3], [0, 1, 2]))

sai: 1  đúng: 6


## 14. Mẹo & nhận dạng pattern

- "Chọn tối đa `k` lần, mỗi lần có tập lựa chọn **mở rộng dần** theo tài nguyên" → **sắp xếp theo ngưỡng + con trỏ + heap**.
- Lựa chọn không làm giảm tài nguyên (lãi `>= 0`) → nghĩ tới **tham lam + lập luận trao đổi**.
- Heap "rổ ứng viên" + con trỏ "mở khóa": cùng khuôn với LC 630 Course Schedule III, LC 871 Minimum Number of Refueling Stops, LC 1353 Maximum Number of Events.
- Luôn kiểm chứng tham lam bằng vét cạn trên input nhỏ ngẫu nhiên trước khi tin.

## 15. Biến thể

1. **Lợi nhuận có thể âm:** tham lam vẫn đúng nếu ta **không bắt buộc** làm đủ `k` (dừng khi lãi lớn nhất `< 0`); nếu bắt buộc làm đúng `k` thì phức tạp hơn.
2. **Dự án tốn vốn** (trừ `capital[i]` rồi cộng `profits[i]`): lúc đó lãi thuần là `profits - capital`, tham lam theo lãi thuần trong số làm được.
3. **Đếm số dự án tối thiểu để đạt vốn mục tiêu `T`** — giống LC 871 (tiếp nhiên liệu).
4. **Trả về danh sách dự án đã chọn** — lưu chỉ số trong heap.

Demo biến thể 3 — số dự án ít nhất để đạt vốn `T` (hoặc -1):

In [26]:
def min_projects_to_reach(T, w, profits, capital):
    projects = sorted(zip(capital, profits)); heap = []; i = 0; cnt = 0
    while w < T:
        while i < len(projects) and projects[i][0] <= w:
            heapq.heappush(heap, -projects[i][1]); i += 1
        if not heap:
            return -1
        w -= heapq.heappop(heap); cnt += 1
    return cnt

print(min_projects_to_reach(4, 0, [1, 2, 3], [0, 1, 1]))    # 2
print(min_projects_to_reach(100, 0, [1, 2, 3], [0, 1, 1]))  # -1
assert min_projects_to_reach(6, 0, [1, 2, 3], [0, 1, 2]) == 3

2
-1


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi thứ tự chọn | `O(n^k)` | `O(k)` | Kiểm chứng tham lam |
| 2. Quét tuyến tính | Tham lam, tìm max bằng vòng lặp | `O(k·n)` | `O(n)` | `n`, `k` nhỏ; dễ hiểu |
| 3. Sắp xếp + heap | Con trỏ mở khóa + max-heap | `O((n+k) log n)` | `O(n)` | Lời giải chuẩn |
| 4. Heap + đường tắt | Như 3, cộng một lần khi đã mở khóa hết | `O((n+k) log n)`, thường nhanh hơn | `O(n)` | `k` rất lớn |

## 17. Bài tập

**Bài 1 (đoán output).** `Solution().findMaximizedCapital(1, 0, [5], [1])` trả về gì?

**Bài 2 (tính tay).** `k = 2, w = 2, profits = [3, 1, 4], capital = [3, 2, 5]`. Trace từng lượt và cho đáp án.

**Bài 3 (điền TODO).** Hoàn thiện vòng lặp chính.

In [27]:
def ipo_todo(k, w, profits, capital):
    projects = sorted(zip(capital, profits))
    heap, i = [], 0
    for _ in range(k):
        # TODO: mở khóa các dự án có vốn yêu cầu <= w (đẩy -profit vào heap)
        # TODO: nếu heap rỗng thì break
        # TODO: lấy lợi nhuận lớn nhất cộng vào w
        pass
    return w

# Không gọi khi chạy notebook. Tự kiểm tra:
# assert ipo_todo(2, 0, [1, 2, 3], [0, 1, 1]) == 4

**Bài 4 (tự viết lại).** Viết lại `Solution` nhưng trả về **danh sách chỉ số dự án** đã làm theo thứ tự.

**Bài 5 (mini challenge).** Biến thể "dự án tốn vốn": làm dự án `i` cần vốn `>= cost[i]`, khi làm **trừ** `cost[i]` rồi nhận về `revenue[i]` (với `revenue[i] >= cost[i]`). Tối đa vốn sau ≤ `k` dự án. Gợi ý: tham lam theo `revenue - cost` trong số làm được — hãy kiểm chứng bằng vét cạn.

In [28]:
def ipo_cost_todo(k, w, cost, revenue):
    # TODO
    raise NotImplementedError

# Không gọi khi chạy notebook.

## 18. Gợi ý 3 mức

**Bài 1–2**
- Mức 1: Bài 1: vốn 0 < 1.
- Mức 2: Bài 2: vốn 2 chỉ làm được dự án 1 (cần 2).
- Mức 3: Bài 2: 2 → 3 (dự án 1) → 6 (dự án 0). Dự án 2 cần vốn 5 nên không kịp mở khóa trong 2 lượt. Đáp án 6.

**Bài 3**
- Mức 1: `while i < len(projects) and projects[i][0] <= w`.
- Mức 2: Nhớ `i += 1`.
- Mức 3: `w -= heapq.heappop(heap)`.

**Bài 4**
- Mức 1: Đẩy `(-profit, index)` vào heap.
- Mức 2: Sắp xếp chỉ số theo vốn: `sorted(range(n), key=lambda i: capital[i])`.
- Mức 3: Kiểm tra lại bằng `simulate(order, ...)`.

**Bài 5**
- Mức 1: Điều kiện làm được vẫn là `cost[i] <= w`.
- Mức 2: Sau khi làm, vốn thay đổi `+ (revenue - cost) >= 0` → vẫn đơn điệu.
- Mức 3: Vậy đây chính là bài gốc với `profits = revenue - cost`, `capital = cost`.

## 19. Đáp án tham khảo

In [29]:
# Bài 1, 2
assert Solution().findMaximizedCapital(1, 0, [5], [1]) == 0
assert Solution().findMaximizedCapital(2, 2, [3, 1, 4], [3, 2, 5]) == 6

# Bài 3
def ipo_sol(k, w, profits, capital):
    projects = sorted(zip(capital, profits))
    heap, i = [], 0
    for _ in range(k):
        while i < len(projects) and projects[i][0] <= w:
            heapq.heappush(heap, -projects[i][1]); i += 1
        if not heap:
            break
        w -= heapq.heappop(heap)
    return w
assert ipo_sol(2, 0, [1, 2, 3], [0, 1, 1]) == 4

# Bài 4
def ipo_order(k, w, profits, capital):
    idx = sorted(range(len(profits)), key=lambda i: capital[i])
    heap, i, order = [], 0, []
    for _ in range(k):
        while i < len(idx) and capital[idx[i]] <= w:
            heapq.heappush(heap, (-profits[idx[i]], idx[i])); i += 1
        if not heap:
            break
        p, j = heapq.heappop(heap)
        w -= p; order.append(j)
    return order, w

random.seed(11)
for _ in range(300):
    n = random.randint(1, 7); k = random.randint(1, 5); w0 = random.randint(0, 6)
    P_ = [random.randint(0, 6) for _ in range(n)]; C_ = [random.randint(0, 12) for _ in range(n)]
    order, wf = ipo_order(k, w0, P_, C_)
    assert simulate(order, w0, P_, C_) == wf == ipo_brute(k, w0, P_, C_)
print("Bài 1–4 OK, ví dụ:", ipo_order(2, 0, [1, 2, 3], [0, 1, 1]))

Bài 1–4 OK, ví dụ: ([0, 2], 4)


In [30]:
# Bài 5
def ipo_cost(k, w, cost, revenue):
    return Solution().findMaximizedCapital(k, w, [r - c for r, c in zip(revenue, cost)], cost)

def ipo_cost_brute(k, w, cost, revenue):
    n = len(cost); best = w; used = [False] * n
    def dfs(left, cur):
        nonlocal best
        best = max(best, cur)
        if left == 0:
            return
        for i in range(n):
            if not used[i] and cost[i] <= cur:
                used[i] = True; dfs(left - 1, cur - cost[i] + revenue[i]); used[i] = False
    dfs(k, w)
    return best

random.seed(12)
for _ in range(400):
    n = random.randint(1, 6); k = random.randint(1, 4); w0 = random.randint(0, 8)
    cost = [random.randint(0, 10) for _ in range(n)]
    revenue = [c + random.randint(0, 6) for c in cost]
    assert ipo_cost(k, w0, cost, revenue) == ipo_cost_brute(k, w0, cost, revenue)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "con trỏ mở khóa + max-heap":**

```python
items = sorted(zip(threshold, value))
heap, i = [], 0
for _ in range(k):
    while i < len(items) and items[i][0] <= resource:
        heappush(heap, -items[i][1]); i += 1
    if not heap: break
    resource -= heappop(heap)          # cộng value lớn nhất
return resource
```

**Checklist trước khi submit:**

- [ ] Sắp dự án theo **vốn yêu cầu**, không theo lợi nhuận.
- [ ] Mở khóa **trong mỗi lượt** (vốn tăng thì mở thêm).
- [ ] Max-heap bằng số âm; nhớ đổi dấu khi cộng.
- [ ] `break` khi heap rỗng.
- [ ] Test `k > n`, không làm được dự án nào, `w` rất lớn, lãi bằng 0.
- [ ] Đã kiểm chứng tham lam bằng vét cạn.